# General tracking-scene visualizer

Browse any extracted tracking-scene **category** and **scene** from one Napari viewer.

This notebook deliberately keeps `napari_scene_visualizer.py` unchanged. It uses the reusable scene-browser widget from that module and adds notebook-level support for loading the latest Stage 8 track paths and reconstructed centers whenever a scene is selected.

Center labels use the source detection reference rather than the persistent tracker identity:

- `C71 | F12` means **cell ID 71 in original frame 12**;
- track IDs remain available in the point-layer properties and continue to define the path layer;
- virtual merge centers retain the merged source cell ID, so two reconstructed centers may legitimately display the same cell ID and frame.

The dock also displays the original frame corresponding to the current local scene time index.


In [ ]:
%gui qt

## Locate the project

In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root from the current working directory."""
    start = (start or Path.cwd()).resolve()

    for candidate in (start, *start.parents):
        visualizer_file = (
            candidate
            / "diagnostics"
            / "tracking_scene_extraction"
            / "napari_scene_visualizer.py"
        )
        scenes_directory = candidate / "data" / "tracking_scenes"

        if visualizer_file.is_file() and scenes_directory.is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not find the project root containing both "
        "'diagnostics/tracking_scene_extraction/napari_scene_visualizer.py' "
        "and 'data/tracking_scenes'."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)


## Configuration

`SHOW_STAGE_8_OVERLAY` controls whether the latest Stage 8 paths and centers are added automatically after loading a scene. A checkbox in the Napari dock allows this to be changed while the viewer is running.

Change `STAGE_8_DIR` only when your current `tracks.csv` is stored elsewhere.

In [ ]:
SCENES_ROOT = PROJECT_ROOT / "data" / "tracking_scenes"

STAGE_8_DIR = (
    PROJECT_ROOT
    / "data"
    / "sample"
    / "processed"
    / "stage_8_track_stitching"
)

TRACKS_PATH = STAGE_8_DIR / "tracks.csv"
TRACKING_METADATA_PATH = STAGE_8_DIR / "metadata.json"

USE_ORIGINAL_COORDINATES = False
SHOW_STAGE_8_OVERLAY = True

print("Scenes root:", SCENES_ROOT)
print("Stage 8 tracks:", TRACKS_PATH)


## Stage 8 scene-track resolution

A saved scene stores the selected `(frame, cell_id)` references. For the currently selected scene, the code below:

1. matches those references against the latest Stage 8 `tracks.csv`;
2. includes repaired merge rows through `source_merged_cell_id` when available;
3. identifies the relevant track IDs; and
4. loads their complete paths over the scene's saved frame interval.

A missing or incompatible Stage 8 result does **not** prevent the extracted scene itself from loading.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


def _parse_boolean_series(series: pd.Series) -> pd.Series:
    """Convert common CSV boolean representations to a boolean Series."""
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin({"true", "1", "yes"})
    )


def load_relevant_stage_8_tracks(
    scene: Any,
    *,
    tracks_path: str | Path,
    metadata_path: str | Path | None = None,
) -> tuple[pd.DataFrame, str, dict[str, Any]]:
    """Load the latest Stage 8 rows relevant to one extracted scene."""
    tracks_path = Path(tracks_path)
    metadata_path = Path(metadata_path) if metadata_path is not None else None

    if not tracks_path.is_file():
        raise FileNotFoundError(
            f"Stage 8 tracks were not found: {tracks_path}. "
            "Run 08_track_stitching.ipynb or update TRACKS_PATH."
        )

    tracks = pd.read_csv(tracks_path)

    required_columns = {"track_id", "frame", "z", "y", "x"}
    missing_columns = required_columns - set(tracks.columns)
    if missing_columns:
        raise ValueError(
            "Stage 8 tracks.csv is missing required column(s): "
            + ", ".join(sorted(missing_columns))
        )

    if metadata_path is not None and metadata_path.is_file():
        with metadata_path.open("r", encoding="utf-8") as file:
            stage_8_metadata = json.load(file)

        stage_8_sample = str(stage_8_metadata.get("sample_id", ""))
        if stage_8_sample and scene.sample_id and stage_8_sample != scene.sample_id:
            raise ValueError(
                "The selected scene and Stage 8 output belong to different samples "
                f"({scene.sample_id!r} versus {stage_8_sample!r})."
            )

    scene_cell_column = str(scene.metadata.get("cell_id_column", "cell_id"))

    if scene_cell_column not in tracks.columns:
        fallback_columns = [
            column
            for column in ("cell_id", "cell")
            if column in tracks.columns
        ]
        if not fallback_columns:
            raise ValueError(
                "Could not find a compatible cell-ID column in Stage 8 tracks.csv. "
                f"Expected '{scene_cell_column}', 'cell_id', or 'cell'."
            )
        scene_cell_column = fallback_columns[0]

    selected_cells = scene.metadata.get("selected_cells", {})
    if not isinstance(selected_cells, dict) or not selected_cells:
        raise ValueError(
            "The selected scene does not contain any saved cell references."
        )

    reference_records = [
        {
            "frame": int(frame),
            "selected_cell_id": int(cell_id),
        }
        for frame, cell_ids in selected_cells.items()
        for cell_id in cell_ids
    ]

    if not reference_records:
        raise ValueError(
            "The selected scene contains an empty selected_cells mapping."
        )

    selected_references = (
        pd.DataFrame(reference_records)
        .sort_values(["frame", "selected_cell_id"])
        .reset_index(drop=True)
    )

    numeric_track_id = pd.to_numeric(tracks["track_id"], errors="coerce")
    numeric_frame = pd.to_numeric(tracks["frame"], errors="coerce")
    numeric_cell_id = pd.to_numeric(
        tracks[scene_cell_column],
        errors="coerce",
    )

    if "source_merged_cell_id" in tracks.columns:
        numeric_source_merged_cell_id = pd.to_numeric(
            tracks["source_merged_cell_id"],
            errors="coerce",
        )
    else:
        numeric_source_merged_cell_id = pd.Series(
            np.nan,
            index=tracks.index,
            dtype=float,
        )

    reference_match = pd.Series(False, index=tracks.index)

    for reference in selected_references.itertuples(index=False):
        frame_match = numeric_frame.eq(int(reference.frame))
        cell_match = numeric_cell_id.eq(int(reference.selected_cell_id))
        source_merge_match = numeric_source_merged_cell_id.eq(
            int(reference.selected_cell_id)
        )
        reference_match |= frame_match & (cell_match | source_merge_match)

    matched_reference_rows = tracks.loc[reference_match].copy()

    if matched_reference_rows.empty:
        raise ValueError(
            "None of the scene's saved frame/cell references were found "
            "in the current Stage 8 tracks.csv."
        )

    scene_track_ids = sorted(
        pd.to_numeric(
            matched_reference_rows["track_id"],
            errors="raise",
        )
        .astype(int)
        .unique()
        .tolist()
    )

    scene_frame_values = {int(frame) for frame in scene.frames}

    scene_tracks = tracks[
        numeric_track_id.isin(scene_track_ids)
        & numeric_frame.isin(scene_frame_values)
    ].copy()

    scene_tracks["track_id"] = pd.to_numeric(
        scene_tracks["track_id"],
        errors="raise",
    ).astype(int)
    scene_tracks["frame"] = pd.to_numeric(
        scene_tracks["frame"],
        errors="raise",
    ).astype(int)

    if "is_virtual_merge" in scene_tracks.columns:
        scene_tracks["is_virtual_merge"] = _parse_boolean_series(
            scene_tracks["is_virtual_merge"]
        )
    else:
        scene_tracks["is_virtual_merge"] = False

    scene_tracks = (
        scene_tracks
        .sort_values(["track_id", "frame"])
        .reset_index(drop=True)
    )

    summary = {
        "selected_reference_count": int(len(selected_references)),
        "matched_reference_count": int(len(matched_reference_rows)),
        "track_ids": scene_track_ids,
        "track_row_count": int(len(scene_tracks)),
        "virtual_center_count": int(scene_tracks["is_virtual_merge"].sum()),
    }

    return scene_tracks, scene_cell_column, summary


## Add the latest paths and source references to Napari

The paths still use stable Stage 8 `track_id` values. Point labels instead show `C<cell_id> | F<original_frame>` so the corresponding detection can be loaded directly from the original processed data.


In [ ]:
STAGE_8_LAYER_PREFIX = "Stage 8 | "


def remove_stage_8_layers(viewer: Any) -> None:
    """Remove only Stage 8 overlay layers created by this notebook."""
    for layer in list(viewer.layers):
        if str(layer.name).startswith(STAGE_8_LAYER_PREFIX):
            viewer.layers.remove(layer)


def _format_integer_identifier(value: Any) -> str:
    """Format a CSV identifier without an unnecessary decimal suffix."""
    numeric = pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]
    if pd.isna(numeric):
        return "?"
    return str(int(numeric))


def add_stage_8_tracking_overlay(
    viewer: Any,
    scene: Any,
    scene_tracks: pd.DataFrame,
    *,
    scene_cell_column: str,
    use_original_coordinates: bool = False,
) -> dict[str, Any]:
    """Add current Stage 8 paths and source-referenced centers to a scene."""
    remove_stage_8_layers(viewer)

    if scene_tracks.empty:
        raise ValueError("There are no Stage 8 track rows to display.")

    frame_to_local_time = {
        int(frame): int(local_time)
        for local_time, frame in enumerate(scene.frames)
    }

    plot_rows = scene_tracks.copy()
    plot_rows["scene_time"] = plot_rows["frame"].map(frame_to_local_time)

    for column in ("z", "y", "x"):
        plot_rows[column] = pd.to_numeric(
            plot_rows[column],
            errors="coerce",
        )

    plot_rows = plot_rows.dropna(
        subset=["scene_time", "z", "y", "x"]
    ).copy()

    if plot_rows.empty:
        raise ValueError(
            "All relevant Stage 8 rows have invalid frame or center coordinates."
        )

    plot_rows["scene_time"] = plot_rows["scene_time"].astype(int)
    plot_rows["frame"] = pd.to_numeric(
        plot_rows["frame"],
        errors="raise",
    ).astype(int)

    # The ordinary Stage 8 cell-ID column is the source detection ID. For a
    # reconstructed merge row, fall back to source_merged_cell_id only when the
    # normal cell-ID value is unavailable.
    source_cell_ids = pd.to_numeric(
        plot_rows[scene_cell_column],
        errors="coerce",
    )

    if "source_merged_cell_id" in plot_rows.columns:
        merged_source_ids = pd.to_numeric(
            plot_rows["source_merged_cell_id"],
            errors="coerce",
        )
        source_cell_ids = source_cell_ids.fillna(merged_source_ids)

    plot_rows["display_cell_id"] = [
        _format_integer_identifier(value)
        for value in source_cell_ids
    ]
    # The visible point label identifies the source cell only. The original
    # frame is shown once in the dock's Scene information section.
    plot_rows["display_frame"] = plot_rows["frame"].astype(str)
    plot_rows["display_label"] = "C" + plot_rows["display_cell_id"]

    if "merge_role" in plot_rows.columns:
        merge_roles = (
            plot_rows["merge_role"]
            .astype(object)
            .where(pd.notna(plot_rows["merge_role"]), "")
            .astype(str)
            .str.strip()
        )
    else:
        merge_roles = pd.Series("", index=plot_rows.index, dtype=str)

    plot_rows["virtual_display_label"] = plot_rows["display_label"]
    has_merge_role = merge_roles.ne("")
    plot_rows.loc[has_merge_role, "virtual_display_label"] = (
        plot_rows.loc[has_merge_role, "display_label"]
        + " | "
        + merge_roles.loc[has_merge_role]
    )

    crop_origin = np.asarray(scene.crop_origin_zyx, dtype=float)
    plot_rows[["scene_z", "scene_y", "scene_x"]] = (
        plot_rows[["z", "y", "x"]].to_numpy(dtype=float)
        - crop_origin
    )

    scale = (1.0, *scene.voxel_size_zyx)

    if use_original_coordinates:
        spatial_translate = tuple(
            origin * spacing
            for origin, spacing in zip(
                scene.crop_origin_zyx,
                scene.voxel_size_zyx,
            )
        )
    else:
        spatial_translate = (0.0, 0.0, 0.0)

    translate = (
        float(scene.frames[0]),
        *spatial_translate,
    )

    track_data = plot_rows[
        [
            "track_id",
            "scene_time",
            "scene_z",
            "scene_y",
            "scene_x",
        ]
    ].to_numpy(dtype=float)

    property_columns = [
        column
        for column in (
            "display_label",
            "virtual_display_label",
            "display_cell_id",
            "display_frame",
            "track_id",
            "frame",
            scene_cell_column,
            "is_virtual_merge",
            "merge_event_id",
            "merge_role",
            "source_track_id",
            "source_merged_cell_id",
        )
        if column in plot_rows.columns
    ]

    def property_values(series: pd.Series) -> np.ndarray:
        values = series.astype(object)
        values = values.where(pd.notna(values), "")
        return values.astype(str).to_numpy()

    def point_properties(rows: pd.DataFrame) -> dict[str, np.ndarray]:
        return {
            column: property_values(rows[column])
            for column in property_columns
        }

    created_layers: dict[str, Any] = {}

    # Track geometry still uses track_id; only the visible point labels change.
    created_layers["tracks"] = viewer.add_tracks(
        track_data,
        name=f"{STAGE_8_LAYER_PREFIX}Tracks",
        scale=scale,
        translate=translate,
        tail_length=max(len(scene.frames) + 2, 2),
        tail_width=3,
    )

    ordinary_rows = plot_rows[
        ~plot_rows["is_virtual_merge"].astype(bool)
    ].copy()

    if not ordinary_rows.empty:
        ordinary_point_data = ordinary_rows[
            [
                "scene_time",
                "scene_z",
                "scene_y",
                "scene_x",
            ]
        ].to_numpy(dtype=float)

        created_layers["centers"] = viewer.add_points(
            ordinary_point_data,
            name=(
                f"{STAGE_8_LAYER_PREFIX}Cell centers (C=cell ID)"
            ),
            scale=scale,
            translate=translate,
            size=5,
            face_color="red",
            properties=point_properties(ordinary_rows),
            text={
                "string": "{display_label}",
                "size": 8,
                "color": "white",
                "anchor": "upper_left",
            },
        )

    virtual_rows = plot_rows[
        plot_rows["is_virtual_merge"].astype(bool)
    ].copy()

    if not virtual_rows.empty:
        virtual_point_data = virtual_rows[
            [
                "scene_time",
                "scene_z",
                "scene_y",
                "scene_x",
            ]
        ].to_numpy(dtype=float)

        created_layers["virtual_centers"] = viewer.add_points(
            virtual_point_data,
            name=f"{STAGE_8_LAYER_PREFIX}Virtual merge centers",
            scale=scale,
            translate=translate,
            size=9,
            face_color="yellow",
            properties=point_properties(virtual_rows),
            text={
                "string": "{virtual_display_label}",
                "size": 9,
                "color": "white",
                "anchor": "upper_left",
            },
        )

    return created_layers


## Notebook-level general scene browser

This subclass composes the existing reusable `TrackingSceneVisualizerWidget`. The original module is not edited. The only additional behavior is the optional Stage 8 overlay and its status display.

In [ ]:
from qtpy.QtWidgets import (
    QCheckBox,
    QGroupBox,
    QLabel,
    QVBoxLayout,
)

from diagnostics.tracking_scene_extraction.napari_scene_visualizer import (
    TrackingSceneVisualizerWidget,
    add_scene_to_viewer,
    load_tracking_scene,
)


class GeneralTrackingSceneVisualizerWidget(TrackingSceneVisualizerWidget):
    """Scene picker plus notebook-owned latest-track overlay support."""

    def __init__(
        self,
        *,
        viewer: Any,
        scenes_root: str | Path,
        tracks_path: str | Path,
        tracking_metadata_path: str | Path | None = None,
        use_original_coordinates: bool = False,
        show_stage_8_overlay: bool = True,
    ) -> None:
        self.tracks_path = Path(tracks_path)
        self.tracking_metadata_path = (
            Path(tracking_metadata_path)
            if tracking_metadata_path is not None
            else None
        )
        self._initial_show_stage_8_overlay = bool(show_stage_8_overlay)
        super().__init__(
            viewer=viewer,
            scenes_root=scenes_root,
            use_original_coordinates=use_original_coordinates,
        )

    def _build_ui(self, use_original_coordinates: bool) -> None:
        super()._build_ui(use_original_coordinates)

        root_layout = self.layout()

        overlay_group = QGroupBox("Latest tracking result")
        overlay_layout = QVBoxLayout()
        overlay_group.setLayout(overlay_layout)

        self.stage_8_checkbox = QCheckBox(
            "Show latest Stage 8 paths and centers"
        )
        self.stage_8_checkbox.setChecked(
            self._initial_show_stage_8_overlay
        )
        overlay_layout.addWidget(self.stage_8_checkbox)

        self.overlay_status_label = QLabel(
            "Load a scene to resolve its current tracks."
        )
        self.overlay_status_label.setWordWrap(True)
        overlay_layout.addWidget(self.overlay_status_label)

        root_layout.insertWidget(root_layout.count() - 1, overlay_group)

    def _connect_events(self) -> None:
        super()._connect_events()
        self.stage_8_checkbox.toggled.connect(
            self._on_stage_8_toggled
        )
        self.viewer.dims.events.current_step.connect(
            self._on_viewer_step_changed
        )

    def _on_viewer_step_changed(self, _event: Any | None = None) -> None:
        if self.current_scene is not None:
            self._update_info(self.current_scene)

    def _current_scene_frame_values(
        self,
        scene: Any,
    ) -> tuple[str, str]:
        """Return the current scene time index and original frame."""
        if len(scene.frames) == 0:
            return "-", "-"

        try:
            local_time = int(round(float(self.viewer.dims.current_step[0])))
        except Exception:
            return "unavailable", "unavailable"

        if not 0 <= local_time < len(scene.frames):
            return str(local_time), "outside saved scene"

        return str(local_time), str(int(scene.frames[local_time]))

    def _update_info(self, scene: Any) -> None:
        """Update the standard Scene information panel, including frame."""
        selected_cells = scene.metadata.get("selected_cells", {})
        selected_count = sum(len(values) for values in selected_cells.values())
        first_frame = int(scene.frames[0]) if len(scene.frames) else "-"
        last_frame = int(scene.frames[-1]) if len(scene.frames) else "-"
        shape = tuple(int(value) for value in scene.instance_labels.shape[1:])
        scene_time, original_frame = self._current_scene_frame_values(scene)

        self.info_label.setText(
            f"Category: {scene.category}\n"
            f"Scene: {scene.name}\n"
            f"Sample: {scene.sample_id}\n"
            f"Current scene time index: {scene_time}\n"
            f"Current original frame: {original_frame}\n"
            f"Saved frames: {first_frame}–{last_frame} ({len(scene.frames)})\n"
            f"Selected cell entries: {selected_count}\n"
            f"Crop shape ZYX: {shape}\n"
            f"Crop origin ZYX: {scene.crop_origin_zyx}\n"
            f"Voxel size ZYX: {scene.voxel_size_zyx}"
        )

    def load_selected_scene(self) -> None:
        """Load the selected saved scene and refresh its current tracks."""
        try:
            scene = load_tracking_scene(self.selected_scene_path())
            add_scene_to_viewer(
                self.viewer,
                scene,
                use_original_coordinates=(
                    self.original_coordinates_checkbox.isChecked()
                ),
                remove_existing=True,
            )
        except Exception as error:
            self._show_error(str(error))
            return

        self.current_scene = scene
        self._update_info(scene)
        overlay_message = self._refresh_stage_8_overlay()

        self.status_label.setText(
            f"Loaded {scene.category}/{scene.name}. {overlay_message}"
        )

    def _reload_current_scene(self, _checked: bool) -> None:
        """Reload the current scene after changing coordinate mode."""
        if self.current_scene is None:
            return

        try:
            add_scene_to_viewer(
                self.viewer,
                self.current_scene,
                use_original_coordinates=(
                    self.original_coordinates_checkbox.isChecked()
                ),
                remove_existing=True,
            )
        except Exception as error:
            self._show_error(str(error))
            return

        self._update_info(self.current_scene)
        overlay_message = self._refresh_stage_8_overlay()
        self.status_label.setText(
            f"Reloaded {self.current_scene.category}/"
            f"{self.current_scene.name}. {overlay_message}"
        )

    def _on_stage_8_toggled(self, checked: bool) -> None:
        if not checked:
            remove_stage_8_layers(self.viewer)
            self.overlay_status_label.setText(
                "Latest Stage 8 overlay is disabled."
            )
            return

        if self.current_scene is None:
            self.overlay_status_label.setText(
                "Load a scene to resolve its current tracks."
            )
            return

        message = self._refresh_stage_8_overlay()
        self.status_label.setText(message)

    def _refresh_stage_8_overlay(self) -> str:
        remove_stage_8_layers(self.viewer)

        if self.current_scene is None:
            message = "No scene is loaded."
            self.overlay_status_label.setText(message)
            return message

        if not self.stage_8_checkbox.isChecked():
            message = "Latest Stage 8 overlay is disabled."
            self.overlay_status_label.setText(message)
            return message

        try:
            scene_tracks, cell_column, summary = (
                load_relevant_stage_8_tracks(
                    self.current_scene,
                    tracks_path=self.tracks_path,
                    metadata_path=self.tracking_metadata_path,
                )
            )

            add_stage_8_tracking_overlay(
                self.viewer,
                self.current_scene,
                scene_tracks,
                scene_cell_column=cell_column,
                use_original_coordinates=(
                    self.original_coordinates_checkbox.isChecked()
                ),
            )
        except Exception as error:
            message = f"Stage 8 overlay unavailable: {error}"
            self.overlay_status_label.setText(message)
            return message

        track_ids = ", ".join(
            str(track_id) for track_id in summary["track_ids"]
        )
        message = (
            f"Stage 8 overlay loaded: {summary['track_row_count']} rows, "
            f"{len(summary['track_ids'])} track(s) [{track_ids}], "
            f"{summary['virtual_center_count']} virtual center(s). "
            "Center labels use C=cell ID; the current original frame is shown "
            "in Scene information."
        )
        self.overlay_status_label.setText(message)
        return message

    def clear_scene(self) -> None:
        remove_stage_8_layers(self.viewer)
        super().clear_scene()
        self.overlay_status_label.setText(
            "Load a scene to resolve its current tracks."
        )


## Launch the general visualizer

Use the **Category** and **Scene** selectors in the right-side dock, then press **Load scene**. Press **Refresh** after extracting a new scene while the viewer is open.

Center labels follow `C<cell_id> | F<original_frame>`. The **Frame lookup** section maps the current local scene time index back to the original frame number.


In [ ]:
import napari


viewer = napari.Viewer(
    ndisplay=3,
    title="General Tracking Scene Visualizer",
)

scene_browser = GeneralTrackingSceneVisualizerWidget(
    viewer=viewer,
    scenes_root=SCENES_ROOT,
    tracks_path=TRACKS_PATH,
    tracking_metadata_path=TRACKING_METADATA_PATH,
    use_original_coordinates=USE_ORIGINAL_COORDINATES,
    show_stage_8_overlay=SHOW_STAGE_8_OVERLAY,
)

viewer.window.add_dock_widget(
    scene_browser,
    area="right",
    name="General Tracking Scene Visualizer",
)

print(
    "General scene visualizer opened.\n"
    "Choose a category and scene in the dock, then click 'Load scene'."
)
